In [3]:
import numpy as np
import pandas as pd
from numpy.linalg import norm
from pathlib import Path

def proj_psd(A: np.ndarray) -> np.ndarray:
    w, V = np.linalg.eigh(A)
    A_psd = (V * np.maximum(w, 0.0)) @ V.T
    return 0.5 * (A_psd + A_psd.T)

def higham_nearcorr(A: np.ndarray,
                    tol: float | None = None,
                    max_iterations: int = 100,
                    weights: np.ndarray | None = None) -> np.ndarray:
    if not np.allclose(A, A.T, atol=1e-12):
        raise ValueError("Input matrix must be symmetric.")
    n = A.shape[0]
    eps = np.finfo(float).eps
    if tol is None:
        tol = eps * n
    if weights is None:
        weights = np.ones(n)
    W12 = np.sqrt(np.outer(weights, weights))

    X = A.copy()
    Y = A.copy()
    D = np.zeros_like(A)

    rel_diffX = rel_diffY = rel_diffXY = np.inf
    it = 0
    while max(rel_diffX, rel_diffY, rel_diffXY) > tol:
        it += 1
        if it > max_iterations:
            break

        X_old = X.copy()
        R = X - D
        X = proj_psd(W12 * R) / W12
        D = X - R

        Y_old = Y.copy()
        Y = X.copy()
        np.fill_diagonal(Y, 1.0)

        nY = norm(Y, "fro") + eps
        rel_diffX  = norm(X - X_old, "fro") / (norm(X, "fro") + eps)
        rel_diffY  = norm(Y - Y_old, "fro") / nY
        rel_diffXY = norm(Y - X, "fro") / nY

        X = Y.copy()
    return X

def to_correlation(M: np.ndarray) -> np.ndarray:
    M = 0.5 * (M + M.T)
    d = np.diag(M).copy()
    d = np.where(d < 1e-18, 1e-18, d)
    sd = np.sqrt(d)
    return (M / sd[:, None]) / sd[None, :]

# Read data and give output
DATA_DIR = Path.cwd() / "testfiles_" / "data"
csv_path = DATA_DIR / "testout_1.4.csv"

R = pd.read_csv(csv_path)
for c in R.columns:
    R[c] = pd.to_numeric(R[c], errors="coerce")
if R.shape[0] != R.shape[1]:
    raise ValueError(f"Matrix must be square, got {R.shape}")
R.index = R.columns

A = R.to_numpy(float)
# convert to correlation if not already
if not np.allclose(np.diag(A), 1.0, atol=1e-12):
    A = to_correlation(A)

R_higham = higham_nearcorr(A, max_iterations=200, tol=1e-10)
R_higham_df = pd.DataFrame(R_higham, index=R.index, columns=R.columns)
print(R_higham_df)

          x1        x2        x3        x4        x5
x1  1.000000 -0.483199 -0.241787 -0.067767 -0.714761
x2 -0.483199  1.000000  0.015446  0.405660  0.178286
x3 -0.241787  0.015446  1.000000  0.488250  0.336248
x4 -0.067767  0.405660  0.488250  1.000000 -0.322136
x5 -0.714761  0.178286  0.336248 -0.322136  1.000000
